<h1 align="left">
  <picture>
    <source media="(prefers-color-scheme: dark)" srcset="https://kesai.eu/py123d/_static/123D_logo_transparent_white.svg" width="500">
    <source media="(prefers-color-scheme: light)" srcset="https://kesai.eu/py123d/_static/123D_logo_transparent_black.svg" width="500">
    <img alt="Logo" src="https://kesai.eu/py123d/_static/123D_logo_transparent_black.svg" width="500">
  </picture>
  <h2 align="left">123D: Scene Tutorial</h1>
</h1>

In [ ]:
from typing import Optional

import matplotlib.pyplot as plt
import numpy as np

from py123d.api import SceneAPI, SceneFilter

## 1.1 Download Demo Logs

You can download demo logs for 123D as described in the [documentation](https://kesai.eu/py123d/installation/). After the installation and download, you can start with the tutorial.

## 1.2 Create Scenes by filtering the datasets



The logs store continuous driving recordings. Scenes in 123D are sequences that are extracted from a log, e.g. given a predefined future and history duration or at different sampling frequencies.

In the example below, we filter some scenes from all logs with 1 second duration and 8 seconds temporal distance (making the scenes non-overlapping). If `None` is passed to the duration, the scene will contain the complete log.

This `SceneFilter` object is passed to a `SceneBuilder` object to query `SceneAPI`'s from the dataset.





In [ ]:
from py123d.api import get_filtered_scenes
from py123d.datatypes import CameraID

scene_filter = SceneFilter(
    split_names=None,
    log_names=None,
    scene_uuids=None,
    target_iteration_duration_s=0.1,  # 10Hz iteration frequency
    future_duration_s=1.0,  # Look up to 1 second into the future.
    history_duration_s=0.5,  # Look up to 0.5 seconds into the past.
    required_scene_modalities=["ego_state_se3"],
)
scenes = get_filtered_scenes(scene_filter)

dataset_splits = set(scene.log_metadata.split for scene in scenes)
print(f"Found {len(scenes)} scenes from {len(dataset_splits)} datasplits:")
for split in dataset_splits:
    print(f" - {split}")

## 1.2 Inspecting the Scene

Let's inspect a random scene from our dataset.

A scene has several different metadata objects attached to it:

`SceneMetadata`: Information how the scene was extracted from the log. Each timestep in the log has universally unique identifier (UUID). The UUID of the initial timestep also serves as identifier for scene filtering.

In [ ]:
scene: SceneAPI = np.random.choice(scenes)  # type: ignore
scene_metadata = scene.scene_metadata
print(scene_metadata)
print("\nInitial UUID:", scene_metadata.initial_uuid)
print("Number of iterations:", scene_metadata.num_future_iterations)
print("Number of history iterations:", scene_metadata.num_history_iterations)
print("Duration (s):", scene_metadata.future_duration_s)
print("Iteration duration (s):", scene_metadata.iteration_duration_s)

`LogMetadata`: Information of the log the scene was extracted from. This object also includes data about the map (if available), or other static log values.

In [ ]:
log_metadata = scene.get_log_metadata()
log_metadata

`MapMetadata`: If the map is available, this object includes information about the location, wether the map is 3D (`map_has_z`), of the the map is defined per log (`map_is_per_log`).

In [ ]:
map_metadata = scene.get_map_metadata()
map_metadata

## 1.3 Retrieving data from the `SceneAPI`

### Modality Hierarchy

A scene is composed of **modalities**, different streams of sensor data and annotations. Each modality is identified by:

- A **modality type** (`ModalityType`): the kind of data — `EGO_STATE_SE3`, `CAMERA`, `LIDAR`, `BOX_DETECTIONS_SE3`, `TRAFFIC_LIGHT_DETECTIONS`
- An optional **modality ID**: a specific sensor within a type, e.g. `CameraID.PCAM_F0` (front camera) or `LidarID.LIDAR_TOP`

Some modalities (like ego state) have a single instance per type. Others (like cameras and lidars) can have multiple sensors, each recording independently at different frequencies. The timestamp plot below visualizes this: dashed lines show the iteration grid, while colored ticks show when each modality has recorded data.

In [ ]:
import pandas as pd

from py123d.api.scene.arrow.utils.scene_builder_utils import infer_iteration_duration_from_timestamps_us

rows = []
for key, meta in scene.get_all_modality_metadatas().items():
    modality_timestamps = scene.get_all_modality_timestamps(meta.modality_type, meta.modality_id)
    iteration_duration_s = infer_iteration_duration_from_timestamps_us(np.array(modality_timestamps))
    rows.append(
        {
            "Modality Type": meta.modality_type.serialize(),
            "Modality ID": str(meta.modality_id.serialize()) if meta.modality_id is not None else "n/a",
            "Key": key,
            "Hz": round(1 / iteration_duration_s, 0) if iteration_duration_s > 0 else "n/a",
        }
    )

df = pd.DataFrame(rows)
print(f"This scene has {len(df)} modalities:")
df

In [ ]:
from py123d.visualization.matplotlib.timestamps import plot_scene_timestamps

print(f"Dataset: {scene.dataset}, Log: {scene.log_name}")
fig, ax = plot_scene_timestamps(scene, include_history=True, time_unit="ms")
plt.show()

### Accessing Modality Data

The `SceneAPI` provides three general methods to access modality data:

| Method | Use case |
|--------|----------|
| `get_all_modality_timestamps(type, id)` | Discover all timestamps where a modality has data |
| `get_modality_at_iteration(iteration, type, id)` | Read data by iteration index (`0` = current, negative = history, positive = future) |
| `get_modality_at_timestamp(timestamp, type, id, criteria)` | Read data at a specific timestamp, with matching criteria: `"exact"`, `"nearest"`, `"forward"`, `"backward"` |

The subsections below also show convenience methods (like `get_ego_state_se3_at_iteration`) that wrap these for common modality types.

In [ ]:
# Example: EgoStateSE3

# 1. All timestamps
ego_state_se3_timestamps = scene.get_all_modality_timestamps(modality_type="ego_state_se3")
print("EgoStateSE3 timestamps:", len(ego_state_se3_timestamps))

# 2. Timestamps at iteration 0
ego_state_se3_at_iteration = scene.get_modality_at_iteration(
    modality_type="ego_state_se3", modality_id=None, iteration=0
)
print("EgoStateSE3 at iteration 0:", ego_state_se3_at_iteration is not None)

# 3. Timestamps at timestamp
ego_state_se3_at_timestamp = scene.get_modality_at_timestamp(
    modality_type="ego_state_se3",
    modality_id=None,
    timestamp=ego_state_se3_timestamps[0],
    criteria="exact",
)
print("EgoStateSE3 at timestamp:", ego_state_se3_at_timestamp is not None)

### 1.3.1 `Timestamp`

Current time step in microseconds.

In [ ]:
from py123d.datatypes.time import Timestamp

iteration = 0
timestamp: Timestamp = scene.get_timestamp_at_iteration(iteration=iteration)
print(f"Time at iteration {iteration}:", timestamp)

### 1.3.2 `EgoStateSE3` 
State of the ego vehicle in 3D with location and orientation.

In [ ]:
ego_state_se3 = scene.get_ego_state_se3_at_iteration(iteration=iteration)

if ego_state_se3 is not None:
    print("Ego parameters\t", ego_state_se3.metadata)

    # The ego vehicles coordinate system is defined by it's rear-axle / IMU location.
    print("Rear axle location:\t", ego_state_se3.rear_axle_se3.point_3d)
    print("Rear axle orientation:\t", ego_state_se3.rear_axle_se3.quaternion)

    # You can also use the center pose
    print("Center location:\t", ego_state_se3.center_se3.point_3d)
    print("Center orientation:\t", ego_state_se3.center_se3.quaternion)

### 1.3.3 `BoxDetectionsSE3`

Object that contains all bounding boxes in the current time step

In [ ]:
from py123d.datatypes.detections.box_detections import BoxDetectionsSE3

box_detections_se3: Optional[BoxDetectionsSE3] = scene.get_box_detections_se3_at_iteration(iteration=iteration)

if box_detections_se3 is not None:
    print(f"Number of boxes:{len(box_detections_se3)}")

    if len(box_detections_se3) > 0:
        box_detection = box_detections_se3[0]
        print("\nFirst box:")
        print("Dataset Label:\t", box_detection.attributes.label)
        print("Default Label:\t", box_detection.attributes.default_label)
        print("Parameters:\t", box_detection.bounding_box_se3)

### 1.3.4 `Camera`
Object containing the camera observation with a pinhole model.

In [ ]:
from typing import Optional

from py123d.datatypes import Camera

available_camera_ids = scene.available_camera_ids
print("Available pinhole camera types:\t", available_camera_ids)

random_camera_id = np.random.choice(available_camera_ids + [CameraID.PCAM_F0])  # type: ignore
random_camera: Optional[Camera] = scene.get_camera_at_iteration(iteration=iteration, camera_id=random_camera_id)


# NOTE: If a camera is not available, the return will be None
if random_camera is not None:
    print(f"\nCamera ID:\t{random_camera_id}")

    print(f"Image shape:\t{random_camera.image.shape}")
    print(f"Model:\t{random_camera.metadata.camera_model}")

    plt.imshow(random_camera.image)
    plt.title(f"Camera ID: {random_camera_id}")
    plt.show()

### 1.3.5 `Lidar`
Object containing a point cloud of a single laser scanner.

In [ ]:
from py123d.datatypes.sensors import LidarID

available_lidar_ids = scene.available_lidar_ids
print("Available Lidar types:\t", available_lidar_ids)


random_lidar_id = np.random.choice(available_lidar_ids + [LidarID.LIDAR_TOP])  # type: ignore
random_lidar = scene.get_lidar_at_iteration(iteration=iteration, lidar_id=random_lidar_id)

if random_lidar is not None:
    point_cloud_3d = random_lidar.point_cloud_3d
    point_cloud_features = random_lidar.point_cloud_features

    print(f"\nLidar ID:\t{random_lidar_id}")
    print(f"Point cloud:\t shape {random_lidar.point_cloud_3d.shape}, dtype: {random_lidar.point_cloud_3d.dtype}")
    print("Features:")
    if random_lidar.point_cloud_features is not None:
        for key in random_lidar.point_cloud_features.keys():
            print(
                f"\t{key}:\t shape: {random_lidar.point_cloud_features[key].shape}\t dtype: {random_lidar.point_cloud_features[key].dtype}"
            )
    else:
        print("\tNo point cloud features available.")

    xy = random_lidar.xy

    plt.scatter(xy[:, 0], xy[:, 1], s=0.1, alpha=0.25, c="black")
    plt.title(f"Lidar ID: {random_lidar_id}")
    plt.xlabel("X-forward [m]")
    plt.ylabel("Y-left [m]")
    plt.axis("equal")

    range_limit = 100  # meters
    plt.xlim(-range_limit, range_limit)
    plt.ylim(-range_limit, range_limit)
    plt.show()

### 1.3.6 `MapAPI`

The `MapAPI` can get retrieved from a scene directly. If the map is available, we plot the map with our default plotting function.
For further information, you can visit the map or visualization tutorial.

In [ ]:
from py123d.api import MapAPI
from py123d.geometry import Point2D
from py123d.visualization.matplotlib.observation import add_default_map_on_ax
from py123d.visualization.matplotlib.utils import add_non_repeating_legend_to_ax


def simple_map_visualization(map_api: MapAPI, center_2d: Point2D, map_radius: float = 100.0):
    """Helper to plot the map using matplotlib.

    :param map_api: The MapAPI to visualize
    :param center_2d: The center point of the map visualization
    :param map_radius: The radius around the center point to visualize
    """

    fsize = 8
    _, ax = plt.subplots(figsize=(fsize, fsize))
    add_default_map_on_ax(ax, map_api=map_api, point_2d=center_2d, radius=map_radius)
    add_non_repeating_legend_to_ax(ax)
    ax.set_aspect("equal")
    ax.set_xlim(center_2d.x - map_radius, center_2d.x + map_radius)
    ax.set_ylim(center_2d.y - map_radius, center_2d.y + map_radius)
    plt.show()


map_api = scene.get_map_api()
if map_api is not None:
    ego_state_se3 = scene.get_ego_state_se3_at_iteration(iteration=iteration)
    if ego_state_se3 is not None:
        center_2d = ego_state_se3.center_se3.point_2d
    else:
        center_2d = Point2D.from_array(np.array([0.0, 0.0]))

    print("\nMapAPI is available.")
    print("Map Metadata:", map_api.map_metadata)
    simple_map_visualization(map_api=map_api, center_2d=center_2d)

### 1.3.7 Others:

You can find further modalities in the documentation of [`SceneAPI`](todo)